# Plan B : serveur d'embeddings sur Colab

**À quoi ça sert.** L'application Django calcule déjà à l'avance les vecteurs des 232 passages (fichiers `.npy`). Pour une
recherche avec embeddings, il ne reste qu'à calculer le vecteur de la **question**, avec le **même modèle** que le notebook.
Le PC du binôme n'a pas assez de mémoire pour charger ces modèles, et le service d'inférence de Hugging Face peut refuser
l'appel (jeton, crédits). Ce notebook fait donc tourner **les deux mêmes modèles** (multilingual-e5-small et LaBSE) sur Colab
et les rend joignables par une adresse temporaire : l'application envoie la question, Colab renvoie le vecteur.

```
Question -> application Django (PC) -> adresse temporaire (tunnel Cloudflare) -> ce notebook (Colab) -> vecteur
```

**Mode d'emploi**
1. Exécuter les cellules **dans l'ordre** (aucun fichier à téléverser, aucun compte Hugging Face ni jeton : les modèles sont publics).
2. La dernière cellule affiche l'adresse du serveur, une **clé secrète** et les commandes à copier sur le PC.
3. Sur le PC, dans le dossier `application_django`, copier les commandes affichées, puis `python manage.py preparer_en_ligne`
   (il vérifie que les vecteurs sont les mêmes que ceux du notebook) et `python manage.py runserver`.
4. **Laisser cette cellule tourner et l'onglet Colab ouvert** pendant toute la démonstration.

**À savoir**
- L'adresse et la clé changent à chaque exécution : après un redémarrage de Colab, refaire l'étape 3.
- Une session Colab gratuite s'arrête après une inactivité ou au bout de quelques heures. Lancer ce notebook environ 15 minutes
  avant la démonstration. Comme `preparer_en_ligne` garde les vecteurs des questions d'exemple dans un cache, ces questions
  marchent ensuite même si Colab s'arrête.
- Seul le **texte de la question** passe par Cloudflare et Colab, jamais les documents. La clé empêche un inconnu qui trouverait
  l'adresse de s'en servir. Ne pas publier la clé (ni l'adresse) dans une capture d'écran ou un message.
- Un processeur suffit (pas besoin de GPU). Le chargement des deux modèles prend environ une à trois minutes.

In [1]:
import subprocess
import sys

# mêmes versions que le notebook principal (transformers 5 n'est pas encore compatible avec la suite)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers<5", "sentence-transformers"], check=True)
print("bibliothèques prêtes")

bibliothèques prêtes


## 1. Le serveur

La cellule suivante ne fait que **définir** le serveur (rien ne démarre) : une seule route de calcul, `POST /embeddings`,
protégée par une clé, qui n'accepte que les deux modèles du projet. Le serveur n'utilise que la bibliothèque standard de Python.

In [2]:
"""Serveur d'embeddings du projet 8 (plan B quand le service d'inférence de Hugging Face n'est pas utilisable).

Il fait tourner sur Colab les DEUX modèles du notebook (multilingual-e5-small et LaBSE) et répond à l'application Django :
elle envoie le texte d'une question, le serveur renvoie son vecteur (normalisé, comme dans le notebook).
Aucune bibliothèque à installer en plus de sentence-transformers : le serveur utilise seulement la bibliothèque standard.

Adresses :
  GET  /sante       -> état et dimension de chaque modèle (sans clé : sert à vérifier que le serveur répond)
  POST /embeddings  -> {"modele": "...", "textes": ["...", ...]} avec l'en-tête  Authorization: Bearer <clé>
                       réponse : {"modele": "...", "dimension": 384, "vecteurs": [[...], ...]}
Le préfixe « query: » de e5 est ajouté par l'application, pas par le serveur : le serveur encode le texte tel quel.
"""
import hmac
import json
import queue
import re
import subprocess
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

MODELES_AUTORISES = ("intfloat/multilingual-e5-small", "sentence-transformers/LaBSE")
TEXTES_MAX = 32            # nombre de textes par appel
CARACTERES_MAX = 2000      # longueur d'un texte (un passage du corpus fait moins de 600 caractères)
CORPS_MAX = 100_000        # taille d'une requête, en octets
AGENT = "projet8-recherche/1.0"
ADRESSE_CLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"


def creer_serveur(modeles, cle, hote="127.0.0.1", port=8000):
    """Serveur HTTP prêt à démarrer (serveur.serve_forever()).
    modeles : {nom du modèle: objet SentenceTransformer} ; cle : secret que l'application doit envoyer."""
    inconnus = [nom for nom in modeles if nom not in MODELES_AUTORISES]
    if inconnus:
        raise ValueError(f"modèles non autorisés : {inconnus}")
    verrou = threading.Lock()          # un seul calcul à la fois par le modèle (plusieurs requêtes peuvent arriver ensemble)
    dimensions = {nom: len(m.encode(["test"], normalize_embeddings=True, show_progress_bar=False)[0])
                  for nom, m in modeles.items()}          # sert aussi à « chauffer » les modèles
    attendue = ("Bearer " + cle).encode("utf-8")

    class Gestionnaire(BaseHTTPRequestHandler):
        def repondre(self, code, objet):
            corps = json.dumps(objet, ensure_ascii=False).encode("utf-8")
            self.send_response(code)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(corps)))
            self.end_headers()
            self.wfile.write(corps)

        def log_message(self, format, *args):      # une ligne par appel : heure, méthode, adresse, code (jamais le texte)
            print(time.strftime("%H:%M:%S"), self.command, self.path.split("?")[0], *args[1:2])

        def do_GET(self):
            if self.path.split("?")[0] in ("/", "/sante"):
                self.repondre(200, {"etat": "ok", "modeles": dimensions})
            else:
                self.repondre(404, {"erreur": "adresse inconnue"})

        def do_POST(self):
            if self.path.split("?")[0] != "/embeddings":
                return self.repondre(404, {"erreur": "adresse inconnue"})
            fournie = self.headers.get("Authorization", "").encode("utf-8")
            if not hmac.compare_digest(fournie, attendue):
                return self.repondre(401, {"erreur": "clé absente ou incorrecte"})
            try:
                taille = int(self.headers.get("Content-Length", ""))
            except ValueError:
                return self.repondre(400, {"erreur": "taille de la requête absente"})
            if taille > CORPS_MAX:
                return self.repondre(413, {"erreur": f"requête trop grande (maximum {CORPS_MAX} octets)"})
            try:
                demande = json.loads(self.rfile.read(taille).decode("utf-8"))
                nom, textes = demande["modele"], demande["textes"]
            except (ValueError, KeyError, TypeError):
                return self.repondre(400, {"erreur": "il faut un JSON avec « modele » et « textes »"})
            if nom not in modeles:
                return self.repondre(400, {"erreur": f"modèle inconnu : choisir parmi {sorted(modeles)}"})
            if (not isinstance(textes, list) or not 1 <= len(textes) <= TEXTES_MAX
                    or not all(isinstance(t, str) and 0 < len(t) <= CARACTERES_MAX for t in textes)):
                return self.repondre(400, {"erreur": f"« textes » : de 1 à {TEXTES_MAX} textes non vides de "
                                                     f"{CARACTERES_MAX} caractères au plus"})
            try:
                with verrou:
                    vecteurs = modeles[nom].encode(textes, normalize_embeddings=True, show_progress_bar=False)
            except Exception as erreur:                                   # mémoire insuffisante, etc.
                return self.repondre(500, {"erreur": f"calcul impossible ({type(erreur).__name__})"})
            self.repondre(200, {"modele": nom, "dimension": dimensions[nom],
                                "vecteurs": [[round(float(x), 7) for x in v] for v in vecteurs]})

    return ThreadingHTTPServer((hote, port), Gestionnaire)


def appeler(adresse, chemin, cle=None, donnees=None, delai=30):
    """Appel HTTP comme le fait l'application Django (mêmes en-têtes). Retourne le JSON de la réponse."""
    en_tetes = {"User-Agent": AGENT}
    if donnees is not None:
        en_tetes["Content-Type"] = "application/json"
    if cle:
        en_tetes["Authorization"] = "Bearer " + cle
    corps = None if donnees is None else json.dumps(donnees).encode("utf-8")
    with urllib.request.urlopen(urllib.request.Request(adresse + chemin, data=corps, headers=en_tetes), timeout=delai) as r:
        return json.loads(r.read().decode("utf-8"))


def ouvrir_tunnel(port, delai=90):
    """Rend le serveur joignable depuis Internet avec cloudflared (tunnel « quick » : gratuit, sans compte).
    Retourne (processus, adresse https). L'adresse change à chaque exécution."""
    binaire = Path("cloudflared")
    if not binaire.exists():
        subprocess.run(["wget", "-q", "-O", str(binaire), ADRESSE_CLOUDFLARED], check=True)
        binaire.chmod(0o755)
    processus = subprocess.Popen([str(binaire.resolve()), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"],
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lignes, vues = queue.Queue(), []

    def lire():
        for ligne in processus.stdout:
            lignes.put(ligne)
        lignes.put(None)

    threading.Thread(target=lire, daemon=True).start()
    fin = time.time() + delai
    while time.time() < fin:
        try:
            ligne = lignes.get(timeout=1)
        except queue.Empty:
            continue
        if ligne is None:
            break
        vues.append(ligne.rstrip())
        trouve = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", ligne)
        if trouve:
            return processus, trouve.group(0)
    processus.terminate()
    raise RuntimeError("cloudflared n'a pas donné d'adresse publique. Dernières lignes :\n" + "\n".join(vues[-8:]))


def attendre_adresse(adresse, delai=90):
    """Une adresse toute neuve met quelques secondes à être connue : on réessaie jusqu'à ce que /sante réponde."""
    fin, derniere = time.time() + delai, None
    while time.time() < fin:
        try:
            return appeler(adresse, "/sante", delai=10)
        except Exception as erreur:
            derniere = erreur
            time.sleep(3)
    raise RuntimeError(f"l'adresse publique ne répond pas : {derniere}")


## 2. Charger les deux modèles

Les modèles viennent du Hub de Hugging Face (public, sans jeton). Un avertissement « HF_TOKEN absent » est normal et sans conséquence.

In [3]:
from sentence_transformers import SentenceTransformer

MODELES = {}
for nom in MODELES_AUTORISES:
    MODELES[nom] = SentenceTransformer(nom)
    print(f"{nom} : {MODELES[nom].get_sentence_embedding_dimension()} dimensions, longueur max {MODELES[nom].max_seq_length} tokens")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (18 outputs left out)

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

sentence-transformers/LaBSE : 768 dimensions, longueur max 256 tokens


/tmp/ipykernel_1038/3893900870.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"{nom} : {MODELES[nom].get_sentence_embedding_dimension()} dimensions, longueur max {MODELES[nom].max_seq_length} tokens")


## 3. Démarrer le serveur et ouvrir l'adresse publique

Cette cellule : crée une clé secrète, démarre le serveur, ouvre le tunnel, **se teste elle-même par l'adresse publique**
(comme le fera l'application), puis affiche les commandes pour le PC et reste active. Pour l'arrêter : bouton « interrompre »,
puis la cellule 4 si on veut couper aussi le serveur et le tunnel.

In [4]:
import secrets

# si la cellule est relancée, on coupe d'abord l'ancien serveur et l'ancien tunnel
if globals().get("serveur") is not None:
    serveur.shutdown()
    serveur.server_close()
if globals().get("tunnel") is not None:
    tunnel.terminate()

CLE = secrets.token_urlsafe(24)
PORT = 8000
serveur = creer_serveur(MODELES, CLE, port=PORT)
threading.Thread(target=serveur.serve_forever, daemon=True).start()
tunnel, ADRESSE = ouvrir_tunnel(PORT)
print("tunnel ouvert, attente de l'adresse publique...")
etat = attendre_adresse(ADRESSE)
print("état du serveur par l'adresse publique :", etat)

# test de bout en bout : un vrai calcul, appelé par l'adresse publique avec la clé
for nom, dimension in (("intfloat/multilingual-e5-small", 384), ("sentence-transformers/LaBSE", 768)):
    texte = "query: frais de scolarité" if "e5" in nom else "frais de scolarité"
    reponse = appeler(ADRESSE, "/embeddings", CLE, {"modele": nom, "textes": [texte]})
    assert reponse["dimension"] == dimension and len(reponse["vecteurs"][0]) == dimension, nom
    print(f"test OK : {nom} -> vecteur de {dimension} dimensions")

print("\n" + "=" * 78)
print("SUR LE PC, dans le dossier application_django :")
print("\n  PowerShell :")
print(f'    $env:PROJET8_SERVEUR = "{ADRESSE}"')
print(f'    $env:PROJET8_CLE = "{CLE}"')
print("\n  cmd :")
print(f"    set PROJET8_SERVEUR={ADRESSE}")
print(f"    set PROJET8_CLE={CLE}")
print("\n  puis (dans la même fenêtre) :")
print("    python manage.py preparer_en_ligne")
print("    python manage.py runserver")
print("=" * 78)
print("Laisser cette cellule active pendant la démonstration.")

try:
    while True:
        time.sleep(30)
        if tunnel.poll() is not None:
            print("Le tunnel s'est arrêté : relancer cette cellule (l'adresse et la clé changeront).")
            break
except KeyboardInterrupt:
    print("Cellule interrompue : le serveur continue en arrière-plan. La cellule 4 l'arrête.")

tunnel ouvert, attente de l'adresse publique...
18:32:43 GET /sante 200
état du serveur par l'adresse publique : {'etat': 'ok', 'modeles': {'intfloat/multilingual-e5-small': 384, 'sentence-transformers/LaBSE': 768}}
18:32:43 POST /embeddings 200
test OK : intfloat/multilingual-e5-small -> vecteur de 384 dimensions
18:32:43 POST /embeddings 200
test OK : sentence-transformers/LaBSE -> vecteur de 768 dimensions

SUR LE PC, dans le dossier application_django :

  PowerShell :
    $env:PROJET8_SERVEUR = "https://<adresse-temporaire>.trycloudflare.com"
    $env:PROJET8_CLE = "<clé affichée par Colab>"

  cmd :
    set PROJET8_SERVEUR=https://<adresse-temporaire>.trycloudflare.com
    set PROJET8_CLE=<clé affichée par Colab>

  puis (dans la même fenêtre) :
    python manage.py preparer_en_ligne
    python manage.py runserver
Laisser cette cellule active pendant la démonstration.
Cellule interrompue : le serveur continue en arrière-plan. La cellule 4 l'arrête.


## 4. Arrêter (facultatif)

In [5]:
# Arrêter le serveur et le tunnel
serveur.shutdown()
serveur.server_close()
tunnel.terminate()
print("serveur et tunnel arrêtés")

serveur et tunnel arrêtés
